# Experimentos controlados — `experiment_v1`

Protocolo preparado em 2026-10-08 e executado pelo autor sobre `de41ee0`.
**Catálogo concluído: três candidatos; referência de 19 features preservada.**

A referência é HGB (**Histogram-based Gradient Boosting**), boosting de árvores
baseado em histogramas, implementado por `HistGradientBoostingClassifier`.
Os próximos marcos estão no [mural de metas](../../docs/project/ROADMAP.md).

O [documento responsável](../../docs/modeling/EXPERIMENT_PROTOCOL.md) registra hipóteses,
resultados e limites. O [contrato JSON](../../references/experiment_protocol_v1.json)
continua sendo a política congelada antes dos fits. As células de código abaixo
leem o protocolo e preservam seu estado de execução. Os resultados históricos
em Markdown foram transcritos dos artefatos compartilhados pelo autor.


In [ ]:
import json
from pathlib import Path

import pandas as pd

root = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "references/experiment_protocol_v1.json").is_file()
)
protocol = json.loads((root / "references/experiment_protocol_v1.json").read_text())
protocol["version"], protocol["status"], protocol["budget"]

## Catálogo e hipóteses

Importância negativa por permutação motiva experimentos; não demonstra que um
modelo retreinado melhora. A contagem de rótulos conhecidos e as taxas de fraude
do terminal serão mantidas. Não eliminaremos os indicadores de validade das razões.


In [ ]:
pd.DataFrame(
    {
        "candidate": candidate["id"],
        "features": candidate["feature_count"],
        "removed": ", ".join(candidate["drop_features"]),
        "hypothesis": candidate["hypothesis"],
    }
    for candidate in protocol["candidates"]
)

## Comparação estatística e decisão

Mesmas transações, clientes, datas e política de 100 vagas. Mostrar diferenças
de AP agregada, métricas diárias e sete exclusões de dia sem refit. AP agregada
não é a média das APs diárias. As exclusões medem influência; não são réplicas
independentes, intervalos de confiança ou testes de hipóteses confirmatórios.

Para elegibilidade de desenvolvimento: ganho absoluto de AP ≥ 0,01 e de precisão
diária média nos 100 clientes ≥ 0,02, com integridade e catálogo completo.
Esses limiares são decisões práticas do portfólio, sem utilidade financeira
validada. Revisar influência, erros diários e custo antes de congelar a escolha.
Sem candidato elegível, manter a referência. Nenhuma promoção em produção.


## Execução e revisão — 2026-10-08

Run `21ccedf10d944092ba874153c1d21257`, revisão `de41ee0`, 18 artefatos verificados
localmente pelo autor. São três runs MLflow independentes, sem refit da referência
ou ajuste na validação. [Recibo](../../references/evidence/controlled_ablation_execution_2026-10-08.json).

| Modelo | Features | AP global | Precisão média @100 | Ocorrências fraudulentas de cliente/dia priorizadas |
| --- | ---: | ---: | ---: | ---: |
| Referência | 19 | 0,623949 | 54,00% | 378 |
| Sem volume do terminal | 17 | 0,610552 | 53,43% | 374 |
| Sem contagens de fraude | 17 | 0,626240 | 53,29% | 373 |
| Sem ambos os grupos | 15 | 0,628595 | 53,14% | 372 |

São 700 vagas e 511 ocorrências fraudulentas de cliente/dia. Os resultados
não contam pessoas únicas nem perdas financeiras evitadas. As ablações deixam
4, 5 e 6 ocorrências adicionais fora da prioridade, respectivamente.

A retirada de volume piora AP nas sete exclusões de dia. A retirada de contagens
muda o sinal do pequeno efeito de AP em duas exclusões. O candidato de 15 features
mantém o sinal positivo, mas seu ΔAP cai de +0,004647 para cerca de +0,00047 ao
retirar 6 ou 11 de maio. Todas as ablações pioram precisão operacional nas sete
exclusões; amplitudes não são intervalos de confiança.

**Decisão:** conservar o HGB de 19 features e encerrar o catálogo de
`experiment_v1`. Nenhum candidato atingiu os ganhos mínimos congelados.
Não houve promoção em produção ou declaração formal de superioridade.
O teste final continua reservado. [Análise completa](../../docs/modeling/EXPERIMENT_PROTOCOL.md#resultados-e-decisão--2026-10-08).

Esta seção registra arquivos enviados pelo autor, não a execução deste notebook.
Modelos, Parquets de scores, manifesto, estado e auditoria continuam no ambiente
local e devem ser preservados. O [runbook](../../docs/operations/EXPERIMENT_EXECUTION.md)
explica a leitura dos resultados existentes sem novos fits.
